# Introduction à Python
## Notebook 04 — Décomposer un problème avec des fonctions
**Cycle ingénieur · 4e année · Séance de 3 heures**

Ce notebook est votre cours, votre terrain d'expérimentation et votre support de révision. Nous reprenons le challenge du NB03 non réalisé en classe, puis nous apprenons à organiser un programme en traitements que l'on peut nommer, réutiliser et vérifier séparément.

**Objectifs :** définir et appeler une fonction ; distinguer paramètres et arguments ; expliquer `return` ; décomposer un cahier des charges ; composer plusieurs traitements ; choisir des tests qui peuvent révéler une erreur.

**Méthode : comprendre la demande → prévoir → écrire → exécuter → expliquer.** Avant une cellule d'exemple, prenez le temps de prédire. Les cellules vides accueillent vos réponses ou votre code. Les points de reprise se déplient après votre tentative ; un corrigé séparé permet de reprendre les exercices après le cours.

Ces durées incluent lecture, manipulation et mise en commun. Les activités **⭐ / ⭐⭐ / ⭐⭐⭐** sont hors de ces 180 minutes : elles servent aux plus rapides ou à la révision. Pour tenir le rythme, les prédictions se discutent brièvement en binôme ; les §7 et §9 conservent leur temps de recherche.

### Avant de commencer
Kernel **Python 3**, cellules exécutées de haut en bas. Aucun fichier ni bibliothèque nécessaire. Les fonctions des §5–6 seront utilisées ensuite : après un redémarrage, réexécutez leurs définitions. Les exemples réinitialisent leurs propres données. Modifier une cellule sans l'exécuter ne change pas la mémoire du kernel.

Nous utilisons les acquis NB01–NB03 : affectations, nombres, booléens, conditions, `for`, `range`, listes, indices, compteurs et accumulateurs. Les moyennes et maxima portent sur des **listes non vides de nombres** ; chaque exercice précise ses hypothèses. Pas de nouvelle structure de données : une fonction renverra ici une seule valeur, éventuellement une liste dans une extension.

## 1. Reprendre le challenge NB03
### Surveiller le compartiment du rover
**12 minutes de recherche, puis 6 minutes de correction/synthèse.** Allez le plus loin possible ; nous mettrons ensuite en commun une solution complète.

Le rover transporte un matériel dont la température doit rester entre **18 et 26 °C, bornes incluses**. Les mesures successives, en °C, sont :

`[20, 17, 23, 28, 26, 18, 30, 22]`

Écrivez un programme qui :
- affiche chaque mesure hors plage ;
- donne le nombre d'anomalies ;
- calcule la moyenne et le maximum ;
- affiche « Relevé conforme » si aucune anomalie n'a été rencontrée, sinon « Inspection nécessaire ».

La liste est non vide. Les bornes doivent pouvoir être modifiées sans réécrire tous les tests. Calculez somme et maximum avec vos étapes, **sans `sum()`, `min()` ni `max()`**. Un ou plusieurs parcours sont possibles.

**Avant de coder :** qu'initialiser ? Que mettre à jour pour chaque mesure ? Que faire une seule fois à la fin ? Écrivez ces trois étapes en français.

<details>
<summary>Point de reprise — après votre tentative</summary>

| Mesures | Anomalies | Moyenne | Maximum | Bilan |
|---|---:|---:|---:|---|
| Liste initiale | 3 : 17, 28, 30 | 23 | 30 | Inspection nécessaire |
| `[18, 26]` | 0 | 22 | 26 | Relevé conforme |
| `[17]` | 1 | 17 | 17 | Inspection nécessaire |

### Une solution monolithique
```python
temperatures_c = [20, 17, 23, 28, 26, 18, 30, 22]
minimum_c = 18
maximum_autorise_c = 26
nb_anomalies = 0
total_c = 0
maximum_c = temperatures_c[0]
for temperature_c in temperatures_c:
    if temperature_c < minimum_c or temperature_c > maximum_autorise_c:
        print("Hors plage :", temperature_c, "°C")
        nb_anomalies += 1
    total_c += temperature_c
    if temperature_c > maximum_c:
        maximum_c = temperature_c
moyenne_c = total_c / len(temperatures_c)
print("Anomalies :", nb_anomalies)
print("Moyenne :", moyenne_c, "°C ; maximum :", maximum_c, "°C")
if nb_anomalies == 0:
    print("Relevé conforme")
else:
    print("Inspection nécessaire")
```
Le compteur ne change que hors plage ; la somme inclut **toutes** les mesures. Le maximum commence sur une mesure réelle. Après quatre passages : compteur 2, somme 88, maximum 28. La division et le bilan arrivent après la boucle.

**Transition : le programme fonctionne, mais comment mieux l'organiser ?**

</details>

## 2. Pourquoi découper un programme ?
Dans la solution précédente, une même boucle mélange comptage, somme, maximum et affichage. Pour changer la règle de diagnostic, nous devons retrouver les bonnes lignes parmi les calculs. Pour utiliser la moyenne ailleurs, nous devons isoler ce qui la produit.

Une **fonction** regroupe un traitement nommé. Elle a une **responsabilité identifiable** : on doit pouvoir dire ce qu'elle fait en une phrase précise. Les données reçues constituent ses **entrées** ; le résultat transmis au code qui l'appelle constitue sa **sortie**. Un affichage est une action visible, mais ne fournit pas à lui seul une valeur réutilisable.

Repartons du programme du §1, sans encore écrire de `def` :

| Responsabilité | Données nécessaires | Résultat utile |
|---|---|---|
| Compter les anomalies | mesures, bornes | entier |
| Calculer la moyenne | mesures non vides | nombre en °C |
| Chercher le maximum | mesures non vides | nombre en °C |
| Produire le diagnostic | nombre d'anomalies | texte |

Le programme principal organisera les appels et les affichages. Séparer ces responsabilités peut conduire à plusieurs parcours de la petite liste : c'est acceptable ici. Nous privilégions des traitements faciles à expliquer et à tester ; découper ne rend pas automatiquement un programme plus rapide.

### À vous — Découper sur papier
Dans la solution monolithique, identifiez les lignes correspondant à chaque responsabilité. Pourquoi la fonction qui choisit le diagnostic n'a-t-elle pas besoin de recevoir toute la liste ? Si la borne supérieure change, quelles entrées changent et quels calculs restent identiques ?

## 3. Définir et appeler une fonction
**Définir** décrit un traitement ; **appeler** demande son exécution. Voici d'abord une fonction qui affiche les dépassements d'un seuil. Nous isolons une responsabilité déjà connue : parcourir puis sélectionner.

Dans `def afficher_depassements(mesures, seuil):` :
- `def` annonce une définition ; `afficher_depassements` est le nom choisi ;
- `mesures` et `seuil` sont les noms des données attendues ;
- les parenthèses et le deux-points délimitent l'en-tête ;
- le bloc indenté est le **corps**. La boucle y est à quatre espaces, le `if` à huit et son affichage à douze.

### Prédire avant d'exécuter
Écrivez tous les affichages dans leur ordre. À quel moment commence le parcours ? La ligne `def` affiche-t-elle quelque chose ?

In [ ]:
print("A")
def afficher_depassements(mesures, seuil):
    for mesure in mesures:
        if mesure > seuil:
            print("Dépassement :", mesure)
print("B")
afficher_depassements([18, 27, 25, 31], 25)
print("C")

### Suivre le flux d'exécution
1. Python affiche A, puis exécute la définition : le nom désigne désormais la fonction. Le corps n'est pas exécuté.
2. Python affiche B, puis rencontre l'appel. Les données de cet appel deviennent disponibles sous les noms `mesures` et `seuil` dans le corps.
3. Le parcours et ses tests s'exécutent ; seuls 27 et 31 sont affichés.
4. À la fin du corps, Python reprend après l'appel et affiche C.

Dans un notebook, la cellule de définition doit avoir été exécutée avant l'appel. Après modification du corps, **réexécutez la définition**, puis l'appel : les anciens résultats visibles ne mettent pas la fonction à jour.

### Manipuler
Appelez la fonction avec `[25, 26, 24]` et un seuil de 25, puis de 26. Prédisez le nombre de lignes produites à chaque appel. Ne recopiez pas le corps.

<details>
<summary>Point de reprise — après votre tentative</summary>

Le critère est strict : 26 dépasse 25 mais ne dépasse pas 26. Aucun affichage au second appel ne signifie pas que le corps a été ignoré : la boucle a examiné les trois valeurs et chaque test était faux.

</details>

## 4. Paramètres et arguments
Un **paramètre** est un nom dans la définition. Un **argument** est la valeur fournie à un appel. Dans nos appels, les arguments sont associés aux paramètres **par leur position**.

```python
releve_c = [18, 27, 25, 31]
limite_c = 25
afficher_depassements(releve_c, limite_c)
```

| Dans l'appel | Dans cet appel du corps |
|---|---|
| valeur de `releve_c` | `mesures` désigne la liste |
| valeur de `limite_c` | `seuil` vaut 25 |

Les noms utilisés à l'extérieur peuvent différer des paramètres. C'est ce qui permet d'employer le même traitement pour plusieurs relevés. Les variables créées dans la fonction sont **locales à l'appel** : elles ne deviennent pas des variables du programme principal. Nous transmettrons les résultats par `return`, au §5. Nous ne modifions pas les listes reçues dans le socle.

### Lire puis généraliser
Le compartiment a maintenant une plage autorisée. Complétez le schéma ci-dessous **dans la cellule vide**, puis appelez la fonction sur le relevé du §1. Les bornes sont incluses : être hors plage signifie être trop bas **ou** trop haut.

```python
def afficher_hors_plage(mesures, minimum, maximum):
    for mesure in mesures:
        if ...:
            print("Hors plage :", mesure)
```

Écrivez ensuite un second appel sur `[18, 26]` et les bornes 18, 26. Indiquez pour ce second appel les trois paramètres et les trois arguments.

<details>
<summary>Point de reprise — après votre tentative</summary>

Le premier appel affiche 17, 28, 30 ; le second n'affiche rien. `mesure < minimum and mesure > maximum` ne conviendrait pas : avec des bornes ordonnées, une valeur ne peut pas être simultanément au-dessous du minimum et au-dessus du maximum. Les bornes sont des paramètres ; les écrire en dur dans le corps ferait perdre la généralité.

</details>

## 5. `return` : transmettre un résultat
### Afficher n'est pas renvoyer
`print` écrit un message pour le lecteur. `return expression` évalue l'expression, **termine immédiatement l'appel en cours** et transmet sa valeur au code appelant. L'appel peut alors prendre place dans un calcul ou une affectation, comme `len(mesures)`.

Une fonction qui atteint la fin de son corps sans `return` transmet la valeur spéciale **`None`** : elle ne transmet pas de résultat de calcul. `None` n'est ni zéro, ni une chaîne vide. Un `print` ne remplace pas un `return`.

Les deux fonctions ci-dessous comptent les mêmes dépassements. Avant d'exécuter, prévoyez tous les affichages ainsi que les valeurs de `a` et `b`.

In [ ]:
def afficher_nombre_anomalies(mesures, seuil):
    compteur = 0
    for mesure in mesures:
        if mesure > seuil:
            compteur += 1
    print(compteur)

def compter_anomalies(mesures, seuil):
    compteur = 0
    for mesure in mesures:
        if mesure > seuil:
            compteur += 1
    return compteur

a = afficher_nombre_anomalies([18, 27, 25, 31], 25)
b = compter_anomalies([18, 27, 25, 31], 25)
print("Résultats :", a, b)

### Comment se déroule `b = compter_anomalies(...)` ?
Python suspend l'affectation : il doit d'abord évaluer sa partie droite. Pendant l'appel, `compteur` passe de 0 à 0, 1, 1, puis 2. Le `return` transmet 2 ; l'appel est terminé ; **alors seulement** le nom `b` est associé à 2.

Le compteur est initialisé à chaque appel : deux appels successifs n'additionnent pas leurs résultats. Lire `compteur` depuis le programme principal ne permet pas de récupérer cette variable locale. Il faut utiliser la valeur de l'appel.

### Stocker, comparer, calculer, transmettre
Prédisez les quatre lignes avant d'exécuter. La fonction `diagnostic` reçoit un **nombre**, pas une liste. Les règles de ce premier diagnostic sont volontairement simples.

In [ ]:
def diagnostic(nombre_anomalies):
    if nombre_anomalies == 0:
        return "NORMAL"
    return "ALERTE"

mesures_c = [18, 27, 25, 31]
nombre = compter_anomalies(mesures_c, 25)
print(nombre)
print(nombre >= 2)
print(100 * nombre / len(mesures_c), "%")
print(diagnostic(nombre))

<details>
<summary>Point de reprise — après votre tentative</summary>

Les quatre lignes donnent `2`, `True`, `50.0 %`, `ALERTE`. Si le nombre vaut zéro, le premier `return` termine l'appel : le second n'est pas exécuté. Sinon, Python passe au second. Les deux chemins renvoient du texte.

`diagnostic(compter_anomalies(mesures_c, 25))` est aussi possible : l'appel intérieur fournit d'abord 2, puis `diagnostic(2)` produit le texte. Des variables intermédiaires rendent souvent la lecture plus facile.

</details>

### À vous — Trois lectures précises
Sans exécuter :
1. Que vaut `total` après les deux appels ci-dessous ? Pourquoi n'est-ce pas le nombre d'anomalies d'un seul relevé ?
2. Dans `diagnostic(compter_anomalies([25], 25))`, quelle fonction finit en premier ? Quel texte est renvoyé ?
3. Si l'on remplace `return compteur` par `print(compteur)` dans `compter_anomalies`, que deviendrait `nombre` ? Pourquoi `nombre >= 2` ne serait-il plus un calcul valide ?

```python
total = compter_anomalies([26, 27], 25)
total = total + compter_anomalies([24, 25], 25)
```
Écrivez vos réponses, puis vérifiez les deux premiers points dans la cellule de manipulation. Pour le troisième, expliquez l'erreur sans modifier notre fonction de référence.

### Un retour peut arriver trop tôt
Voici une variante erronée. Prévoyez son résultat pour `[10, 30, 40]` et le seuil 25. Tracez uniquement ce qui s'exécute réellement. Que change l'indentation du `return` ?

In [ ]:
def compter_trop_tot(mesures, seuil):
    compteur = 0
    for mesure in mesures:
        if mesure > seuil:
            compteur += 1
        return compteur

print(compter_trop_tot([10, 30, 40], 25))

<details>
<summary>Point de reprise — après votre tentative</summary>

Un `return` dans une boucle quitte **la fonction entière**, pas seulement le passage courant. Pour compter toutes les anomalies, il faut retourner après la boucle. Un retour anticipé peut servir à une autre responsabilité, par exemple chercher si une anomalie existe ; c'est le résultat demandé qui décide de sa place.

</details>

**Pause — 10 minutes.** Avant de reprendre, formulez la différence entre « la fonction affiche 2 » et « l'appel vaut 2 ».

## 6. Construire les fonctions de calcul
Nous avons déjà `compter_anomalies(mesures, seuil)` : elle compte les valeurs **strictement supérieures** à un seuil. Elle ne compte pas toutes les sorties d'une plage ; le nom et le contrat doivent toujours être lus ensemble.

Un **contrat** décrit ce que reçoit une fonction, les hypothèses sur ces données et ce qu'elle renvoie. Pour les fonctions suivantes : liste non vide de nombres, aucune modification de la liste, résultat numérique.

### A. Compléter une moyenne
Recopiez et complétez le schéma dans la cellule vide. Avant le code, dites ce que `total` doit représenter après chaque passage et pourquoi le retour est après la boucle.

```python
def calculer_moyenne(mesures):
    total = ...
    for mesure in ...:
        total += ...
    return ...
```

Testez `[16, 20, 24]` puis `[12]`. N'utilisez pas `sum()`.

### B. Lire un maximum, puis le tracer
La définition suivante est fournie. Prévoyez le candidat au maximum après chaque passage pour `[-8, -3, -6]`. Pourquoi le premier élément convient-il comme valeur initiale, et pas forcément zéro ?

In [ ]:
def trouver_maximum(mesures):
    maximum = mesures[0]
    for mesure in mesures:
        if mesure > maximum:
            maximum = mesure
    return maximum

### C. De la plage au booléen
Écrivez `est_dans_plage(valeur, minimum, maximum)` : elle renvoie un booléen, vrai lorsque la valeur est dans la plage **bornes incluses**. Les bornes sont ordonnées. Une comparaison peut être retournée directement, ou vous pouvez utiliser `if` et deux retours.

Testez 18, 26, puis 17 avec la plage 18–26. Appelez aussi `trouver_maximum([-8, -3, -6])` pour vérifier votre trace précédente.

<details>
<summary>Point de reprise — après votre tentative</summary>

Moyennes : 20.0 et 12.0 ; booléens : True, True, False ; maximum : −3. Les compteurs commencent à 0 événement, les accumulateurs à 0 quantité ; un maximum commence sur un élément réel. Ces initialisations expriment des significations différentes.

</details>

### Définitions de référence pour la suite
Après vos essais, exécutez cette cellule pour disposer de versions communes. Elle remplace les définitions portant les mêmes noms ; conservez vos tentatives au-dessus pour les comparer. Elle n'appelle pas les fonctions.

In [ ]:
def calculer_moyenne(mesures):
    total = 0
    for mesure in mesures:
        total += mesure
    return total / len(mesures)

def est_dans_plage(valeur, minimum, maximum):
    return valeur >= minimum and valeur <= maximum

## 7. Décomposer un problème complet
### Cahier des charges : un nouveau diagnostic du compartiment
Nous reprenons les mesures `[20, 17, 23, 28, 26, 18, 30, 22]` °C. Les bornes acceptables sont 18 et 26 °C incluses. Le bilan doit afficher le nombre de valeurs hors plage, la moyenne, le maximum et un diagnostic :
- **« PRIORITAIRE »** si le maximum est **strictement supérieur à 29 °C**, ou s'il y a **au moins 3 anomalies** ;
- sinon **« À SURVEILLER »** s'il reste au moins une anomalie ;
- sinon **« CONFORME »**.

Ces règles sont un modèle pédagogique de tri des relevés. La liste est non vide ; les seuils sont des paramètres. La moyenne est affichée mais n'intervient pas dans la décision. La règle prioritaire doit être examinée d'abord.

### 1. Concevoir sans coder
Répondez d'abord : quelles entrées ? Quels résultats ? Quels sous-problèmes ? Que reçoit et que retourne chaque fonction ? Quelles fonctions existent déjà ?

Reproduisez et complétez ce tableau dans la cellule de réponse ; **pas de boucle avant cette étape**.

| Responsabilité / nom choisi | Entrées et hypothèses | Sortie et type | Réutilisée ou à écrire ? |
|---|---|---|---|
| … | … | … | … |

Ajoutez un pseudo-code du programme principal : ordre des appels, stockage des résultats, affichage. Le diagnostic reçoit les résultats utiles, pas toute la liste.

### 2. Implémenter
Écrivez seulement les fonctions manquantes et le programme principal. Réutilisez les fonctions existantes. Pour le comptage hors plage, utilisez `est_dans_plage` ; pour la décision, utilisez les résultats déjà calculés. Les fonctions de calcul et de décision renvoient des valeurs ; le programme principal les affiche.

### 3. Expliquer
Sur le jeu initial, indiquez dans quel ordre les fonctions se terminent et quelles valeurs passent au diagnostic. Pourquoi `compter_anomalies(mesures, 26)` ne suffit-il pas à compter les anomalies demandées ?

<details>
<summary>Point de reprise — après votre tentative</summary>

Le résultat attendu est **3 ; 23.0 °C ; 30 °C ; PRIORITAIRE**. Pour `[18, 26]`, le bilan doit être CONFORME ; pour `[17, 20]`, À SURVEILLER. Plusieurs décompositions sont possibles, mais les entrées et sorties doivent permettre de vérifier chaque responsabilité séparément.

</details>

## 8. Tests, cas limites et débogage
Un **test** confronte un résultat calculé à un résultat attendu déterminé **avant l'exécution**. « Le programme ne plante pas » ne suffit pas. Un test qui passe n'est pas une preuve pour toutes les entrées ; choisissez des cas capables de distinguer un algorithme correct d'un algorithme plausible mais faux.

### Prévoir, puis vérifier
Complétez les attentes, puis testez les fonctions de référence. Justifiez ce que chaque cas vérifie.

| Appel | Résultat attendu à prévoir | Intérêt |
|---|---|---|
| `compter_anomalies([25], 25)` | … | égalité au seuil |
| `compter_anomalies([26, 25, 27], 25)` | … | plusieurs événements |
| `calculer_moyenne([12])` | … | une seule valeur |
| `trouver_maximum([-8, -3, -6])` | … | toutes les valeurs négatives |
| `compter_anomalies([], 25)` | … | parcours vide |

`assert condition` demande que la condition soit vraie : si elle l'est, rien ne s'affiche ; sinon Python signale une `AssertionError` et interrompt la cellule. C'est une aide à la vérification pendant le développement. Exemple : `assert compter_anomalies([25], 25) == 0`. Écrivez vos vérifications avec `assert` ou des affichages comparés à vos attentes.

### Liste vide : respecter le contrat
Le comptage sur `[]` renvoie 0 : le compteur initial n'a pas changé. En revanche, notre moyenne diviserait par zéro et notre maximum tenterait de lire un premier élément absent. Leurs contrats exigent donc une liste non vide. Nous ne choisissons pas arbitrairement une moyenne ou un maximum de zéro.

Un programme principal peut tester `len(mesures) == 0` et afficher « Aucune mesure » **avant** les appels. Aucun traitement d'exception n'est nécessaire pour ce choix simple. Les nombres décimaux pouvant être approchés, les tests ci-dessus emploient des résultats exacts ; pour comparer une moyenne comme −17/3, vous pouvez comparer les résultats arrondis avec `round(..., 6)`.

### Diagnostiquer un bug logique
Ce programme s'exécute mais ne respecte pas le cahier des charges : compter **toutes** les valeurs strictement supérieures au seuil. Lisez-le avant de lancer la cellule.

In [ ]:
def compter_suspect(mesures, seuil):
    compteur = 0
    for mesure in mesures:
        compteur = 0
        if mesure > seuil:
            compteur += 1
    return compteur

print(compter_suspect([30, 10, 40], 25))

1. Donnez le résultat attendu puis le résultat réel. Tracez le compteur après l'initialisation, puis après chaque passage.
2. Quelle phrase décrivant l'état du compteur devrait rester vraie après chaque passage ? Où cesse-t-elle d'être vraie ?
3. Corrigez dans la cellule vide et testez deux anomalies ainsi qu'aucune anomalie.
4. Pourquoi `[30]` serait-il un test insuffisant pour révéler le bug ?

**Démarche :** résultat attendu → premier écart dans la trace → instruction responsable → correction ciblée → nouveau test. Modifier des lignes au hasard rend l'origine de l'erreur plus difficile à comprendre.

## 9. Challenge autonome — Qualification d'un essai moteur
**5 min de conception, 17 min d'implémentation, 6 min de tests et d'explication.**

Un banc d'essai relève la puissance électrique d'un moteur toutes les **10 secondes**. Chaque valeur, en W, est supposée constante sur un intervalle de 10 s. Le relevé initial est :

`[80, 120, 100, 160, 140]`

### Cahier des charges
Le programme doit produire :
- la puissance moyenne et la puissance maximale, en W ;
- le nombre de mesures **strictement supérieures à 130 W** ;
- l'énergie totale consommée, en Wh : pour chaque intervalle, énergie = puissance × durée en secondes / 3600 ;
- un diagnostic : **« ESSAI À REVOIR »** si l'énergie est **strictement supérieure à 2 Wh** ou s'il y a **au moins 2 dépassements** ; sinon **« ESSAI ACCEPTÉ »**.

Les valeurs sont numériques et positives ou nulles ; la durée d'un intervalle est strictement positive. La liste est non vide dans le socle. Durée et seuils doivent pouvoir changer sans modifier les corps des fonctions. L'égalité à 130 W ne compte pas comme dépassement ; l'égalité à 2 Wh ne suffit pas à refuser l'essai.

### Exigences de réalisation
- Proposer **au moins quatre fonctions aux responsabilités identifiables**, réutilisées ou nouvelles ; au moins deux fonctions doivent être écrites par vous pour ce challenge.
- Séparer les calculs et la décision des affichages ; les fonctions de traitement renvoient leurs résultats. Le programme principal relie les appels.
- Employer des boucles explicites pour les agrégations, sans `sum()`, `min()` ni `max()` ; ne pas modifier la liste reçue.
- Avant le code : écrire les entrées/sorties et la décomposition retenue. Vous choisissez les noms et la répartition des responsabilités.
- Fournir **au moins trois jeux de test**, avec résultats attendus déterminés avant exécution. Ils doivent couvrir une acceptation, un refus dû au nombre de dépassements seul et un refus dû à l'énergie seule. Vérifier aussi au moins une égalité à un seuil.

Une solution qui fonctionne uniquement sur le relevé initial ne suffit pas. Expliquez en quelques phrases pourquoi vos tests distinguent les deux causes de refus.

### Vos tests
Écrivez vos attentes en commentaire avant les appels. Comparez les résultats numériques **et** le diagnostic. Un relevé avec un seul élément est autorisé ; ne confondez pas durée d'un intervalle et durée totale de l'essai.

<details>
<summary>Point de reprise — après votre tentative</summary>

Pour le relevé initial : moyenne **120 W**, maximum **160 W**, **2** dépassements, énergie **6000/3600 ≈ 1.6667 Wh**, donc **ESSAI À REVOIR**, à cause du nombre de dépassements. Pour la même liste à 20 s par intervalle, l'énergie double, pas la moyenne ni le nombre de dépassements.

Pour juger votre décomposition : chaque fonction a-t-elle des entrées explicites et une sortie utilisable ? Pouvez-vous vérifier l'énergie sans passer par l'affichage du bilan ? Une modification de seuil demande-t-elle seulement de changer un argument ?

</details>

### ⭐ Relevé absent
Adaptez le programme principal : avec `[]`, afficher « Aucune mesure » sans appeler moyenne ni maximum. Gardez leurs contrats inchangés. Testez ensuite `[0]`, qui est un vrai relevé avec une mesure nulle.

### ⭐⭐ Localiser les dépassements
Écrivez une fonction qui renvoie la liste des **indices** des mesures strictement au-dessus du seuil, dans l'ordre. Utilisez `range(len(mesures))`, l'indexation et `append()`. La liste initiale doit rester intacte. Sur le relevé initial, attendez `[3, 4]` ; sur `[130]`, `[]`. Expliquez pourquoi les indices diffèrent des numéros de mesure comptés à partir de 1.

### ⭐⭐⭐ Un dépassement isolé ou prolongé ?
Écrivez une fonction renvoyant la **longueur de la plus longue série consécutive** de mesures strictement au-dessus du seuil. Sans nouvelle notion, choisissez les états à mémoriser et expliquez leur signification avant de coder.

Tests au seuil 130 : `[140, 150, 100, 160]` → 2 ; `[100, 130]` → 0 ; `[140, 150, 160]` → 3 ; `[]` → 0. Ne comptez pas seulement le nombre total de dépassements.

## 10. À retenir
| Idée | Ce qu'il faut savoir expliquer |
|---|---|
| Fonction | Un traitement nommé, avec une responsabilité identifiable. |
| Définition / appel | `def` définit ; l'appel déclenche le corps. |
| Paramètres / arguments | Noms attendus dans la définition / valeurs fournies à l'appel. |
| État local | Les variables de travail appartiennent à l'appel ; initialiser au bon endroit. |
| `return` | Termine l'appel et transmet une valeur ; sa position change l'algorithme. |
| `print` | Affiche ; ne fournit pas à lui seul un résultat de calcul réutilisable. |
| Composition | Le résultat d'une fonction devient l'entrée d'un autre traitement. |
| Décomposition | Entrées → résultats recherchés → responsabilités → contrats → code. |
| Test | Prévoir un résultat, exécuter, comparer et expliquer un écart. |

**Nous cherchons désormais à raisonner avant de coder.** Une boucle n'est pas un point de départ automatique : elle répond à un sous-problème identifié.

<details>
<summary>Point de reprise — après votre tentative</summary>

1. Un seul dépassement strict : `1`, puis `ALERTE`.
2. Le total doit représenter toutes les mesures avant la division.
3. Identifier données/hypothèses, résultats attendus et sous-problèmes avec leurs entrées/sorties.

### Sauvegarde
Notez les activités à reprendre. Redémarrez le kernel et exécutez votre notebook dans l'ordre ; vérifiez les jeux de test. Une cellule vide qui ne produit pas d'erreur n'est pas un exercice validé. Enregistrez le fichier et conservez votre travail dans le dépôt habituel du cours.

</details>